# ПРАКТИЧЕСКОЕ ЗАНЯТИЕ 4. ФАЙЛЫ И ИСКЛЮЧЕНИЯ

Выполнила: Софронова Аксинья Романовна

Группа: ИНБ-б-о-26-2

Вариант: 7

## Задание 1

In [3]:
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("Первая строка\nВторая строка\nТретья строка")
with open("sample.txt", "r", encoding="utf-8") as f:
    content_str = f.read()
    print("--- Способ 1: read() ---")
    print(repr(content_str))
    # Комментарий: read() считывает весь файл целиком и возвращает ОДНУ общую строку (str),
    # включая все символы переноса строки '\n'. Удобно для небольших файлов.

print("\n--- Способ 2: цикл for по строкам ---")
with open("sample.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(repr(line))
    # Комментарий: цикл построчно считывает файл (каждая итерация возвращает очередную строку str
    # с символом '\n' в конце). Файл не загружается в память целиком.

with open("sample.txt", "r", encoding="utf-8") as f:
    lines_list = f.readlines()
    print("\n--- Способ 3: readlines() ---")
    print(lines_list)
    # Комментарий: readlines() считывает весь файл и возвращает СПИСОК строк (list[str]),
    # где каждый элемент списка — отдельная строка файла с '\n' на конце.

--- Способ 1: read() ---
'Первая строка\nВторая строка\nТретья строка'

--- Способ 2: цикл for по строкам ---
'Первая строка\n'
'Вторая строка\n'
'Третья строка'

--- Способ 3: readlines() ---
['Первая строка\n', 'Вторая строка\n', 'Третья строка']


## Задание 2

In [4]:
def safe_read_number(filename):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            content = f.read().strip()
            if not content:
                print(f"[{filename}] Ошибка: файл пуст.")
                return None
            first_item = content.split()[0]
            number = float(first_item)
            return int(number) if number.is_integer() else number
    except FileNotFoundError:
        print(f"[{filename}] Ошибка: файл не найден.")
        return None
    except ValueError:
        print(f"[{filename}] Ошибка: первый элемент файла не является числом.")
        return None

# --- Демонстрация работы всех веток ---

with open("valid_num.txt", "w", encoding="utf-8") as f:
    f.write("42 остальной текст")

with open("invalid_num.txt", "w", encoding="utf-8") as f:
    f.write("привет 123")

with open("empty.txt", "w", encoding="utf-8") as f:
    f.write("   \n  ")

print("Результат 1:", safe_read_number("valid_num.txt"))

print("Результат 2:", safe_read_number("no_such_file.txt"))

print("Результат 3:", safe_read_number("invalid_num.txt"))

print("Результат 4:", safe_read_number("empty.txt"))

Результат 1: 42
[no_such_file.txt] Ошибка: файл не найден.
Результат 2: None
[invalid_num.txt] Ошибка: первый элемент файла не является числом.
Результат 3: None
[empty.txt] Ошибка: файл пуст.
Результат 4: None


## Задание 3

In [5]:
import os
from datetime import datetime
def write_log_with_rotation(message, max_lines=5, log_file="log.txt", archive_file="log_1.txt"):
    """Записывает сообщение в лог и выполняет ротацию при достижении max_lines."""
    if os.path.exists(log_file):
        with open(log_file, "r", encoding="utf-8") as f:
            line_count = sum(1 for _ in f)
        if line_count >= max_lines:
            os.replace(log_file, archive_file)
            print(f"[Система] Выполнена ротация: {log_file} -> {archive_file}")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")

# --- Демонстрация работы (лимит N = 3 строки) ---

for fname in ("log.txt", "log_1.txt"):
    if os.path.exists(fname):
        os.remove(fname)
for i in range(1, 6):
    write_log_with_rotation(f"Событие системы №{i}", max_lines=3)
print("\nСодержимое архивного log_1.txt (первые 3 события):")
with open("log_1.txt", "r", encoding="utf-8") as f:
    print(f.read().strip())
print("\nСодержимое актуального log.txt (события 4 и 5):")
with open("log.txt", "r", encoding="utf-8") as f:
    print(f.read().strip())

[Система] Выполнена ротация: log.txt -> log_1.txt

Содержимое архивного log_1.txt (первые 3 события):
[2026-11-25 15:55:54] Событие системы №1
[2026-11-25 15:55:54] Событие системы №2
[2026-11-25 15:55:54] Событие системы №3

Содержимое актуального log.txt (события 4 и 5):
[2026-11-25 15:55:54] Событие системы №4
[2026-11-25 15:55:54] Событие системы №5



Зачем нужна ротация логов в реальных системах:

1. Экономия дискового пространства: работающие годами серверы генерируют гигабайты записей. Без ротации и последующего удаления/сжатия старых архивов диск переполнится, что приведёт к отказу всей системы.
2. Производительность и скорость анализа: открывать, читать и искать ошибки в файле на 5 МБ намного быстрее и проще, чем в едином файле размером 50 ГБ.
3. Удобство резервного копирования и соблюдение требований безопасности: архивные логи за конкретный период легче отправлять в холодное хранилище.

## Ответы на контрольные вопросы

1. • 'r' (read): открывает файл только для чтения. Если файла не существует, возникает ошибка FileNotFoundError.
	• 'w' (write): открывает файл для записи. Если файла нет — создаёт его; если файл существует — полностью стирает его старое содержимое.
	• 'a' (append): открывает файл для добавления данных в самый конец. Если файла нет — создаёт новый; старое содержимое сохраняется.
2. Конструкция with — это менеджер контекста. Она гарантирует, что файл будет автоматически закрыт (f.close()) сразу после выхода из блока кода, даже если внутри блока произошла ошибка. Это защищает от утечки системных дескрипторов и потери несохранённых данных из буфера.
3. При открытии в режиме 'w' файл мгновенно очищается, и в нём останется только новая записанная строка.
Чтобы сохранить старые данные, нужно открывать файл в режиме дозаписи 'a'.
4. write() работает с сырыми потоками символов и записывает ровно то, что ему передали, давая программисту полный контроль над форматированием.
Чтобы записать несколько строк, нужно явно добавлять символ \n в конец каждой строки при вызове write("текст\n") либо передать список строк с \n в метод writelines(["строка 1\n", "строка 2\n"]).
5. • read() считывает весь файл целиком в одну строку (str).
	• readline() считывает ровно одну текущую строку до символа \n (str).
	• readlines() считывает весь файл целиком и возвращает список всех строк (list[str]).
	• Для большого файла предпочтительнее построчная итерация (for line in f:) или readline(), так как они не загружают весь файл в оперативную память сразу.
6. Символ \n физически присутствует в файле как разделитель строк, поэтому Python считывает его как часть строки.
При сравнении: "admin\n" == "admin" вернёт False, поэтому перед проверкой строку очищают методом .strip().
При выводе: функция print() сама добавляет перевод строки в конце, поэтому print(line) даст двойной отступ (пустую строку между строками). Чтобы этого избежать, используют print(line.strip()) или print(line, end="").
7. Файловый объект является ленивым итератором: при цикле for line in f: Python считывает с диска небольшой буфер, отдаёт в переменную line только одну текущую строку, а после обработки заменяет её следующей. В памяти одновременно находится лишь одна строка.
Метод readlines() вынужден выделить память под список и все строки файла одновременно — при файле на 10 ГБ программа упадет с ошибкой нехватки памяти.
8. Переменные-накопители (total = 0, count = 0, freq = {}) обязательно инициализируются до начала цикла. Если объявить их внутри цикла, они будут обнуляться на каждой новой строке.
Внутри цикла значения обновляются: total += num, count += 1, freq[word] = freq.get(word, 0) + 1.
9. Исключение — это объект ошибки, возникающий во время выполнения программы при наступлении нештатной ситуации.
Если исключение не перехвачено блоком try/except, выполнение программы немедленно прерывается, и в консоль выводится трассировка стека (Traceback) с типом и описанием ошибки.
10. • try: код, в котором потенциально может возникнуть ошибка.
	• except: выполняется, если в try произошло указанное исключение.
	• else: выполняется только в том случае, если блок try завершился без единой ошибки.
	• finally: выполняется всегда при любых обстоятельствах.
	• Сценарий для finally: закрытие сетевого соединения, снятие блокировки с ресурса или удаление временного файла после обработки, даже если обработка упала с ошибкой.
11. «Голый» except: перехватывает абсолютно все исключения, включая системные и ошибки самого программиста.
Это маскирует баги в логике кода, делая поиск неисправностей крайне сложным, и мешает пользователю штатно остановить программу.
12. filename = "report.txt"
try:
    with open(filename, "r", encoding="utf-8") as f:
        data = f.read()
except FileNotFoundError:
    print(f"Ошибка: файл '{filename}' не найден. Проверьте имя файла и путь к нему.")
Полезное сообщение должно содержать имя/путь проблемного файла и понятную подсказку, что нужно проверить.
13. • Этап 1 (Чтение): открытие входного файла, фильтрация мусора и извлечение сырых строк (функция read_data(filename)).
	• Этап 2 (Преобразование и расчёт): парсинг строк в числа/структуры, валидация и математические вычисления (функция process_data(raw_items)).
	• Этап 3 (Запись): форматирование итогового отчёта и сохранение в выходной файл (функция save_report(filename, results)).
Разделение на три независимые функции упрощает тестирование логики без привязки к файловой системе.
14. В реальных данных из-за сбоя датчика или опечатки человека могут встречаться битые записи. Если программа упадет на 999-й строке, вся предыдущая работа пропадёт.
Решение: поместить блок try/except ValueError внутри цикла чтения строк, используя enumerate(f, 1) для отслеживания номера строки. При ошибке в except записывать номер и содержимое битой строки в errors.log, а основной цикл продолжать с помощью continue.
15. Проверка: создаётся небольшой тестовый файл с заранее известным, просчитанным вручную ответом. Программа запускается на этом файле, и её вывод сверяется с эталоном.
Почему в отдельный файл: если программа содержит ошибку в алгоритме или прервётся на середине записи в режиме 'w', исходные данные будут безвозвратно уничтожены. Запись в новый файл гарантирует сохранность оригинала и воспроизводимость расчётов.